<a href="https://colab.research.google.com/github/Renan-RodriguesDEV/ChatBots/blob/main/AI_agents_alura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Aula 1. Criando um agente simples


In [ ]:
!pip install -q --upgrade langchain langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.7 MB/s eta 0:00:00


In [ ]:
# variaves de ambiente
from google.colab import userdata
API_KEY = userdata.get('API_KEY')
# imports para o agente do langchain
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# criando o agente (ChatGoogleGenerativeAI), basicamente uma conexão
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0.1,api_key=API_KEY)

In [ ]:
# fazendo uma pergunta (invocando o agente)
response = llm.invoke('Quem é arthur morgan')
print(response.content)

**Arthur Morgan** é o protagonista principal do aclamado jogo de vídeo game **Red Dead Redemption 2**, desenvolvido pela Rockstar Games.

Aqui estão os pontos chave sobre quem ele é:

1.  **Fora-da-lei e Pistoleiro:** Ele é um cowboy e pistoleiro no final do século XIX, um período em que o Velho Oeste americano está em declínio e a civilização está avançando.
2.  **Membro da Gangue Van der Linde:** Arthur é o braço direito de Dutch van der Linde, o carismático e idealista líder da gangue. Ele foi criado por Dutch desde jovem, tornando-se um dos membros mais leais e competentes do grupo.
3.  **Personalidade Complexa:** No início do jogo, Arthur é retratado como um homem pragmático, cínico e muitas vezes brutal, acostumado à vida de fora-da-lei. No entanto, ao longo da história, ele passa por uma profunda jornada de autodescoberta e redenção. Ele é capaz de grande violência, mas também de grande compaixão e lealdade.
4.  **Jornada de Redenção:** Conforme a gangue começa a desmoronar e el

In [ ]:
# criando um prompt para o agente responder com base nesse
# contexto (ele será um atendente de loja de games)
PROMPT = '''
Você é um atendente de uma loja de hardware e jogos, responde-rá apenas perguntas sobre hardware e jogos. Caso contrario, se a pergunta fugir deste escopo, diga: "Desculpe, não posso responder a essa pergunta"
Responda com base em:
- Você é uma android genero feminino, calma, sabia e gentil
- Em caso de perguntas não relacionadas a games e hardware, diga: "Desculpe, não posso responder a essa pergunta, deseja falar com um humano?"
- Resposta ex. em dict do python/json com as chaves:
    {
    "Jogo | Hardware":[
    - descrição: (descricao do jogo ou peça de hardware),
    - preço: (valor em reais ex. R$ 50,00),
    - disponibilidade: (disponível ou indisponível),
    - informações adicionais: (opcional, alguma informação interessante sobre o produto),
    ]
    }
'''

In [ ]:
# criando um padrão de output, saida estruturada
from pydantic import BaseModel, Field
from typing import Literal,Dict,List

# criando um modelo para resposta com pydantic
class ResponseOut(BaseModel):
    descricao:str= Field(title='descricao',max_length=500)
    preco: float
    disponibilidade: Literal['disponível','indisponível']
    informacoes_adicionais:List[str]= Field(title='informacoes adicionais',max_length=500)

In [ ]:
llm_android = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.1,
    api_key=API_KEY
)

In [ ]:
# importando as msg para resposta do sistema e usuario
from langchain_core.messages import HumanMessage,SystemMessage

# cria uma cadeia para o chat responder na estrutura criada (ResponseOut)
android_chain = llm_android.with_structured_output(ResponseOut)

# criando uma função para perguntar
def ask(awnser)->Dict:
    # o tipo de resposta será o ResponseOut
    # e para o chat em cadeia passamos uma lista com o prompt do sistema e a pergunta do humanos
    response:ResponseOut = android_chain.invoke(
        [SystemMessage(content=PROMPT),
        HumanMessage(content=awnser)]
    )
    return response.model_dump()

In [ ]:
perguntas = [
    'quais são os jogos disponiveis?',
    'quais são os hardwares disponiveis?',
    'quais são as especificações da RTX3080?',
    'qual a historia do jogo RDR2?',
    ]

In [ ]:
for pergunta in perguntas:
    response_android = ask(pergunta)
    print(f'HUMAN: {pergunta}\nIA:{response_android}')

## Aula 2. Criando um agente que responde com base em arquivos de contexto

In [ ]:
!pip install -q --upgrade langchain_community faiss-cpu langchain-text-splitters pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
# lib para iterar sobre os arquivos
import os
# importa do langchain community o carregador de PDFs para que possamos carregar o nosso
from langchain_community.document_loaders import PyMuPDFLoader

# lista vazia com docs carregados
pdfs_loaders = []
for arq in os.listdir('.'):
    try:
        if '.pdf' in arq:
            # pra cada PDF carrega e add o PDF a nossa lista de docs
            arq_loader = PyMuPDFLoader(os.path.join('.',arq))
            # passando todos os documentos carregados (.load()) com o extends
            pdfs_loaders.extend(arq_loader.load())
    except Exception as e:
        print('Erro ao carregar o arquivo PDF:', arq)
        print(str(e))

print('PDFs carregados:', len(pdfs_loaders))

PDFs carregados: 15


In [ ]:
# importando o divisor de texto para que possamos dividir o documento
# em chunks (pedaços menores)
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, # tamanho de cada pedaço (300 caracteres)
    chunk_overlap=30 # pega as ultimas 30 letras do chunk anterior
)

# pegando os chunks divididos a partir do nosso splitter e documento
chunks = splitter.split_documents(pdfs_loaders)

In [ ]:
for chunk in chunks:
    print(chunk,'\n\n')

In [ ]:
'''
Embeddings são como traduzir texto para números, onde textos parecidos ficam com números 'próximos'. Isso ajuda a IA a entender e comparar o significado das palavras e frases.
'''

# variaves de ambiente
from google.colab import userdata
API_KEY = userdata.get('API_KEY')
# importando o generative AI embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# conexão com o embedding para imcorporação
embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-001',
    google_api_key=API_KEY
)


In [ ]:
# Criando um vetor de armazenamento a partir dos nossos docs com o FAIS
from langchain_community.vectorstores import FAISS

# Criando o vertor de armazenamento com chunks e nosso embeddings
vectorstore = FAISS.from_documents(documents=chunks,embedding=embeddings)

# definindo o buscador
retriever = vectorstore.as_retriever(
    # tipo de busca, limite da pontuação de similaridade
    search_type='similarity_score_threshold',
    # argumentos de busca, limite de pontuação 0.3
    search_kwargs={'score_threshold':0.3,'k':4}
    )

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
# Criando um llm para responder as questões
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.1,
    api_key=API_KEY
)
# Criando um template de prompt com contexto
prompt_template = ChatPromptTemplate.from_messages(
    [('''
    System: Você será um android que irá tirar duvidas sobre questões envolvendo uma aplicação web de uma padaria.
    Responda SOMENTE com base no contexto fornecido.
    Se não houver resposta no contexto fornecido diga: "Desculpe, não posso responder a essa pergunta"
    '''),
    ('''
    Human: Pergunta: {question}\n\nContexto:\n{context}
    ''')]
)

# criar documentos da cadeia de materiais
document_chain = create_stuff_documents_chain(llm,prompt_template)

In [ ]:

def ask_somenthing(question:str):
    '''Pergunta para o agente de IA'''
    # pegando do buscador documentos relacionados a pergunta
    related_docs = retriever.invoke(question)

    # se não encontrar no documento
    if not related_docs:
        {'answer':'Não sei, desculpe!!',
         'context':[],
         'is_found':False}

    # se encontrou no documento ent. chamamos nossa cadeia de documentos
    answer = document_chain.invoke({
        # passando a pergunta e o contexto que definimos anteriormente
        'question':question,
        'context':related_docs
    })

    return {
        'answer':answer,
        'context':related_docs,
        'is_found':True
    }


In [ ]:
pergunta = 'Qual a ideia centra do artigo e projeto do sistema de padaria?'
response = ask_somenthing(pergunta)

print('Pergunta:',pergunta)
for key in response:
    print(key,':',response[key])

Pergunta: Qual a ideia centra do artigo e projeto do sistema de padaria?
answer : A ideia central do projeto é o desenvolvimento de um sistema de gerenciamento de vendas para padarias, implementado como uma aplicação web, com o intuito de modernizar e otimizar os processos administrativos e financeiros, permitindo aos gestores um controle rigoroso de suas operações.
context : [Document(id='9e2c2f6f-3dd9-400a-91b7-26b5911c5137', metadata={'producer': 'Microsoft® Word para Microsoft 365', 'creator': 'Microsoft® Word para Microsoft 365', 'creationdate': '2025-09-02T13:00:29-03:00', 'source': './Artigo TCC - RenanRodrigues.pdf', 'file_path': './Artigo TCC - RenanRodrigues.pdf', 'total_pages': 15, 'format': 'PDF 1.7', 'title': '', 'author': 'Renan de Souza Rodrigues', 'subject': '', 'keywords': '', 'moddate': '2025-09-02T13:00:29-03:00', 'trapped': '', 'modDate': "D:20250902130029-03'00'", 'creationDate': "D:20250902130029-03'00'", 'page': 3}, page_content='de uma solução robusta, flexível 